In [6]:
import numpy as np
import pandas as pd

symptoms_list = [
    "loss_of_appetite",
    "lethargy",
    "erratic_swimming",
    "isolation",
    "surface_swimming",
    "gasping_for_air",
    "skin_lesions",
    "ulcers",
    "fin_rot",
    "scale_loss",
    "body_swelling",
    "discoloration",
    "eye_cloudiness",
    "excess_mucus",
    "gill_discoloration",
    "rapid_gill_movement",
    "white_spots",
    "fungal_growth",
    "bloody_patches"
]

In [7]:
def assign_disease(row):
    scores = {
        "low_oxygen_stress": 0,
        "bacterial_infection": 0,
        "fungal_infection": 0,
        "parasite_infection": 0,
        "poor_water_quality": 0,
        "healthy": 0
    }

    # -------------------------
    # LOW OXYGEN STRESS
    # -------------------------
    if row["oxygen"] < 4:
        scores["low_oxygen_stress"] += 2
    if row["gasping_for_air"] == 1:
        scores["low_oxygen_stress"] += 2
    if row["rapid_gill_movement"] == 1:
        scores["low_oxygen_stress"] += 1
    if row["surface_swimming"] == 1:
        scores["low_oxygen_stress"] += 1

    # -------------------------
    # BACTERIAL INFECTION
    # -------------------------
    if row["skin_lesions"] == 1:
        scores["bacterial_infection"] += 2
    if row["ulcers"] == 1:
        scores["bacterial_infection"] += 2
    if row["fin_rot"] == 1:
        scores["bacterial_infection"] += 2
    if row["bloody_patches"] == 1:
        scores["bacterial_infection"] += 2
    if row["recent_deaths"] == 1:
        scores["bacterial_infection"] += 1

    # -------------------------
    # FUNGAL INFECTION
    # -------------------------
    if row["fungal_growth"] == 1:
        scores["fungal_infection"] += 3
    if row["discoloration"] == 1:
        scores["fungal_infection"] += 1
    if row["excess_mucus"] == 1:
        scores["fungal_infection"] += 1

    # -------------------------
    # PARASITE INFECTION
    # -------------------------
    if row["white_spots"] == 1:
        scores["parasite_infection"] += 3
    if row["erratic_swimming"] == 1:
        scores["parasite_infection"] += 2
    if row["isolation"] == 1:
        scores["parasite_infection"] += 1

    # -------------------------
    # POOR WATER QUALITY
    # -------------------------
    if row["ph"] < 6 or row["ph"] > 9:
        scores["poor_water_quality"] += 2
    if row["ammonia"] if "ammonia" in row else 0 > 0.5:
        scores["poor_water_quality"] += 2
    if row["lethargy"] == 1:
        scores["poor_water_quality"] += 1

    # -------------------------
    # SEVERITY BOOST (GLOBAL)
    # -------------------------
    if row["death_rate"] > 10:
        for key in scores:
            if key != "healthy":
                scores[key] += 1

    # -------------------------
    # DETERMINE FINAL DISEASE
    # -------------------------
    best_disease = max(scores, key=scores.get)

    # If no strong signal → healthy
    if scores[best_disease] == 0:
        return "healthy"

    return best_disease

In [8]:
data = []

species_options = ["tilapia", "catfish", "trout", "nile_perch"]
age_groups = ["fingerling", "juvenile", "adult"]
water_sources = ["pond", "tank", "cage"]

for _ in range(2000):
    row = {}

    # Basic features
    row["species"] = np.random.choice(species_options)
    row["age_group"] = np.random.choice(age_groups)
    row["temperature"] = np.random.normal(27, 3)
    row["ph"] = np.random.uniform(5.5, 9.5)
    row["oxygen"] = np.random.uniform(2, 8)
    row["stocking_density"] = np.random.randint(5, 50)
    row["water_source"] = np.random.choice(water_sources)
    row["recent_deaths"] = np.random.choice([0, 1])
    row["death_rate"] = np.random.randint(0, 20)

    # Symptoms
    for symptom in symptoms_list:
        row[symptom] = np.random.choice([0, 1], p=[0.7, 0.3])

    # Assign disease
    row["disease"] = assign_disease(row)

    data.append(row)

df = pd.DataFrame(data)
df

,species,age_group,temperature,ph,oxygen,stocking_density,water_source,recent_deaths,death_rate,loss_of_appetite,...,body_swelling,discoloration,eye_cloudiness,excess_mucus,gill_discoloration,rapid_gill_movement,white_spots,fungal_growth,bloody_patches,disease
0,trout,fingerling,27.769973,8.602754,4.418066,43,cage,0,10,0,...,0,1,0,1,0,1,0,1,0,fungal_infection
1,nile_perch,juvenile,24.001247,9.272666,2.705835,17,tank,1,15,1,...,0,1,0,1,0,1,0,0,0,low_oxygen_stress
2,trout,fingerling,27.048274,7.330113,5.491886,12,cage,1,15,1,...,1,0,0,1,1,0,0,0,0,bacterial_infection
3,nile_perch,juvenile,29.317726,9.391784,4.172340,30,pond,0,11,1,...,0,0,1,0,0,0,0,0,0,low_oxygen_stress
4,nile_perch,adult,25.027076,8.267897,4.152431,48,tank,1,19,1,...,1,0,1,0,0,0,0,0,1,bacterial_infection
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,catfish,adult,29.308077,8.072407,4.816082,26,tank,0,2,1,...,1,0,0,0,0,1,0,1,0,bacterial_infection
1996,nile_perch,fingerling,24.063418,5.918499,5.494972,9,cage,0,8,0,...,0,0,1,0,0,0,1,0,0,bacterial_infection
1997,catfish,fingerling,25.215994,8.169024,5.054829,30,pond,1,5,0,...,0,0,0,1,1,0,1,1,0,parasite_infection
1998,catfish,juvenile,29.387362,6.456535,7.811214,41,pond,0,0,0,...,1,0,0,0,1,1,0,0,0,low_oxygen_stress


In [11]:
df.to_csv("../data/fish_disease_dataset.csv", index=False)

In [12]:
df = pd.read_csv("../data/fish_disease_dataset.csv")
df.head()

,species,age_group,temperature,ph,oxygen,stocking_density,water_source,recent_deaths,death_rate,loss_of_appetite,...,body_swelling,discoloration,eye_cloudiness,excess_mucus,gill_discoloration,rapid_gill_movement,white_spots,fungal_growth,bloody_patches,disease
0,trout,fingerling,27.769973,8.602754,4.418066,43,cage,0,10,0,...,0,1,0,1,0,1,0,1,0,fungal_infection
1,nile_perch,juvenile,24.001247,9.272666,2.705835,17,tank,1,15,1,...,0,1,0,1,0,1,0,0,0,low_oxygen_stress
2,trout,fingerling,27.048274,7.330113,5.491886,12,cage,1,15,1,...,1,0,0,1,1,0,0,0,0,bacterial_infection
3,nile_perch,juvenile,29.317726,9.391784,4.172340,30,pond,0,11,1,...,0,0,1,0,0,0,0,0,0,low_oxygen_stress
4,nile_perch,adult,25.027076,8.267897,4.152431,48,tank,1,19,1,...,1,0,1,0,0,0,0,0,1,bacterial_infection
